# Session 2: return, risk and correlation
**A. Real-world problem.** Session 1 had shops that were exactly opposite. Real assets are
only partly opposite. We need numbers: how much does an asset earn on average (its *mean
return*), how much does it wobble (its *standard deviation*, our measure of risk), and how
much do two assets move together (their *correlation* rho, between -1 and +1)? Then: how
should we split money between two assets to get the steadiest result?

**Safety note.** Both assets below are made up (Asset 1: mean 10%, std 20%; Asset 2: mean 6%,
std 10% per year). Returns are drawn from a seeded random generator. This is a model for
learning, NOT investment advice; real markets do not follow a fixed bell curve and past
performance does not predict the future.

**Plan.** (1) simulate returns and measure mean and std, (2) write the two-asset portfolio
formulas and check them on sampled portfolios, (3) see how rho changes the risk, (4) find the
minimum-risk weight and check it by scanning.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

MU1, SD1 = 0.10, 0.20     # asset 1: mean and standard deviation of yearly return
MU2, SD2 = 0.06, 0.10     # asset 2

## Step 1: simulate returns and measure them
We draw two bell-curve (normal) numbers z1, z2 for each simulated year, then

    r1 = mu1 + s1 * z1
    r2 = mu2 + s2 * (rho * z1 + sqrt(1 - rho^2) * z2)

so r1 and r2 have the stated means and standard deviations, and their correlation is rho.
The mean is the average of the yearly returns; the standard deviation is the typical distance
of a year from that average.

In [ ]:
def simulate_returns(n, rho, seed=7, mu1=MU1, s1=SD1, mu2=MU2, s2=SD2):
    """n simulated years of (r1, r2), with correlation rho. Seeded: same seed, same numbers."""
    rng = np.random.default_rng(seed)
    z1 = rng.standard_normal(n)
    z2 = rng.standard_normal(n)
    r1 = mu1 + s1 * z1
    r2 = mu2 + s2 * (rho * z1 + np.sqrt(1 - rho ** 2) * z2)
    return np.stack([r1, r2], axis=1)


def mean_std(x):
    return float(np.mean(x)), float(np.std(x))

## Step 2: the two-asset portfolio
Put weight `w` of the money in asset 1 and `1 - w` in asset 2 (both between 0 and 1).

    expected return  = w mu1 + (1 - w) mu2
    variance         = w^2 s1^2 + (1 - w)^2 s2^2 + 2 w (1 - w) rho s1 s2
    risk (std)       = sqrt(variance)

The return is just a weighted average, so mixing never changes the average in a surprising
way. The variance has an extra last term that depends on rho: this is where diversification
lives.

In [ ]:
def port_return(w, mu1=MU1, mu2=MU2):
    return w * mu1 + (1 - w) * mu2


def port_var(w, rho, s1=SD1, s2=SD2):
    return w ** 2 * s1 ** 2 + (1 - w) ** 2 * s2 ** 2 + 2 * w * (1 - w) * rho * s1 * s2


def port_std(w, rho, s1=SD1, s2=SD2):
    return np.sqrt(port_var(w, rho, s1, s2))


def sampled_portfolio(returns, w):
    """Mean and variance measured on the simulated portfolio returns w r1 + (1 - w) r2."""
    p = w * returns[:, 0] + (1 - w) * returns[:, 1]
    return float(np.mean(p)), float(np.var(p))

## Step 3: the effect of correlation
* rho = +1 (move together): variance = (w s1 + (1 - w) s2)^2, so the risk is just the
  weighted average of the two risks. No benefit.
* rho = -1 (exactly opposite): variance = (w s1 - (1 - w) s2)^2. This is zero when
  w s1 = (1 - w) s2, that is **w = s2 / (s1 + s2)**: the risk can vanish.
* In between, the lower rho, the lower the risk of the mix.

In [ ]:
def zero_variance_weight(s1=SD1, s2=SD2):
    """Weight in asset 1 that makes the variance zero when rho = -1."""
    return s2 / (s1 + s2)

## Step 4: the minimum-variance weight
Differentiate the variance with respect to w and set it to zero:

    w* = (s2^2 - rho s1 s2) / (s1^2 + s2^2 - 2 rho s1 s2)

The bottom is never zero unless s1 = s2 and rho = 1. We check the formula by scanning
many weights and picking the smallest variance. Careful: w* may fall outside 0..1 for some
inputs, which would mean borrowing or short selling; here we only look at 0..1.

In [ ]:
def min_var_weight(rho, s1=SD1, s2=SD2):
    return (s2 ** 2 - rho * s1 * s2) / (s1 ** 2 + s2 ** 2 - 2 * rho * s1 * s2)


def scan_min_var_weight(rho, s1=SD1, s2=SD2, n=100001):
    ws = np.linspace(0.0, 1.0, n)
    return float(ws[np.argmin(port_var(ws, rho, s1, s2))])


def weight_for_return(target, mu1=MU1, mu2=MU2):
    """Weight in asset 1 that gives expected return `target` (solve w mu1 + (1-w) mu2 = target)."""
    return (target - mu2) / (mu1 - mu2)


def max_rho_for_risk(w, risk_limit, s1=SD1, s2=SD2):
    """Largest correlation for which the w-mix has std <= risk_limit (solve the variance for rho)."""
    return (risk_limit ** 2 - w ** 2 * s1 ** 2 - (1 - w) ** 2 * s2 ** 2) / (2 * w * (1 - w) * s1 * s2)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    r = simulate_returns(100000, 0.3)
    for i, name in enumerate(("asset 1", "asset 2")):
        m, s = mean_std(r[:, i])
        print(f"{name}: sample mean {m:.4f}, sample std {s:.4f}")
    print(f"sample correlation {np.corrcoef(r.T)[0, 1]:.4f}")
    for w in (0.0, 0.25, 0.5, 1.0):
        m, v = sampled_portfolio(r, w)
        print(f"w={w:4.2f}: formula return {port_return(w):.4f} std {port_std(w, 0.3):.4f} | "
              f"sampled return {m:.4f} std {np.sqrt(v):.4f}")
    for rho in (1.0, 0.3, 0.0, -0.5, -1.0):
        ws = min_var_weight(rho)
        print(f"rho={rho:5.2f}: w* formula {ws:.4f}, scan {scan_min_var_weight(rho):.4f}, "
              f"std at w* {port_std(np.clip(ws, 0, 1), rho):.4f}")
    wz = zero_variance_weight()
    print(f"rho=-1 zero-variance weight {wz:.4f}: std {port_std(wz, -1.0):.2e}")
    print(f"weight for 7% return {weight_for_return(0.07):.3f}: std at rho 0.3 {port_std(0.25, 0.3):.4f}")
    print(f"max rho for std<=11% at w=0.5: {max_rho_for_risk(0.5, 0.11):.3f}")

    ws = np.linspace(0, 1, 201)
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    for rho, c in [(1.0, "#d9534f"), (0.3, "#e0a030"), (0.0, "#888888"), (-0.5, "#2a9d8f"), (-1.0, "#2a6fdb")]:
        ax.plot(100 * ws, 100 * port_std(ws, rho), color=c, label=f"rho = {rho:g}")
        wm = np.clip(min_var_weight(rho), 0, 1)
        ax.plot(100 * wm, 100 * port_std(wm, rho), "o", color=c)
    ax.set_xlabel("weight in asset 1 (%)")
    ax.set_ylabel("portfolio risk, std of yearly return (%)")
    ax.set_title("Lower correlation, lower risk (dots: minimum risk)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "risk_vs_weight.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6.4, 4.2))
    for rho, c in [(1.0, "#d9534f"), (0.3, "#e0a030"), (-0.5, "#2a9d8f"), (-1.0, "#2a6fdb")]:
        ax2.plot(100 * port_std(ws, rho), 100 * port_return(ws), color=c, label=f"rho = {rho:g}")
    ax2.plot(100 * SD1, 100 * MU1, "ks")
    ax2.plot(100 * SD2, 100 * MU2, "ks")
    ax2.annotate("asset 1", (100 * SD1, 100 * MU1), textcoords="offset points", xytext=(-40, -4), fontsize=8)
    ax2.annotate("asset 2", (100 * SD2, 100 * MU2), textcoords="offset points", xytext=(6, -10), fontsize=8)
    ax2.set_xlabel("risk, std of yearly return (%)")
    ax2.set_ylabel("expected yearly return (%)")
    ax2.set_title("The mix bends to the left as rho falls")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "risk_return_curves.png", dpi=120)

## Where this stops
With more than two assets, writing every pair term by hand gets long. Session 3 uses a
covariance matrix, many assets, and shows what diversification can and cannot remove.

## Coding tasks
1. **Easy.** Print the sampled mean and risk of a portfolio with w = 0.25 and compare with `port_return` and `port_std` (formula).
2. **Stretch.** Use `min_var_weight` for rho = 0.8 and rho = -0.8. Predict which one is larger before you run it, then explain the result.
3. **Explain.** Explain why, when both assets have the same risk (s1 = s2), the minimum-variance weight is 0.5 for every correlation.

## Assessment
- **A. Conceptual.** Why does a correlation of +1 give no diversification benefit, while a correlation of -1 can remove the risk completely?
- **B. Numerical.** Assets 1 and 2 are uncorrelated (rho = 0). Find the weight with the least risk, that risk, and the expected return there.
- **C. Graph and simulation interpretation.** In the first figure, at rho = 0.3 where is the minimum of the risk curve, and is it lower than the risk of the safer asset (10 percent)? What would happen at rho = 1?
- **D. Design and optimisation challenge.** Design: you want an expected return of at least 8 percent and a risk of at most 11 percent from these two assets. What weight do you need and how low must the correlation be? Is rho = 0.3 enough? What can you do if not?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    r = L.simulate_returns(100000, 0.3)
    checks.append(("the simulation is repeatable for a given seed", np.array_equal(r, L.simulate_returns(100000, 0.3))))
    m1, s1 = L.mean_std(r[:, 0])
    m2, s2 = L.mean_std(r[:, 1])
    checks.append((f"sample mean/std of asset 1 are 0.0997 / 0.1997 and of asset 2 are 0.0603 / 0.1001 (got {m1:.4f} {s1:.4f} {m2:.4f} {s2:.4f})",
                   abs(m1 - 0.0997) < 5e-5 and abs(s1 - 0.1997) < 5e-5 and abs(m2 - 0.0603) < 5e-5 and abs(s2 - 0.1001) < 5e-5))
    checks.append(("sample means and stds are close to the stated 10%/20% and 6%/10% (within 0.5 percentage points)",
                   abs(m1 - 0.10) < 0.005 and abs(s1 - 0.20) < 0.005 and abs(m2 - 0.06) < 0.005 and abs(s2 - 0.10) < 0.005))
    c = float(np.corrcoef(r.T)[0, 1])
    checks.append((f"sample correlation is 0.3027, close to rho = 0.3 (got {c:.4f})", abs(c - 0.3027) < 5e-5 and abs(c - 0.3) < 0.01))

    # worked example: w = 0.5, rho = 0.3 -> return 8%, variance 0.0155, std 12.45%
    w, rho = 0.5, 0.3
    var_hand = 0.25 * 0.04 + 0.25 * 0.01 + 2 * 0.25 * 0.3 * 0.02
    checks.append(("worked example: w = 0.5, rho = 0.3 gives return 0.0800, variance 0.0155 and std 0.1245",
                   abs(L.port_return(w) - 0.08) < 1e-12 and abs(L.port_var(w, rho) - 0.0155) < 1e-12
                   and abs(var_hand - 0.0155) < 1e-12 and abs(L.port_std(w, rho) - 0.1245) < 5e-5))
    # the formula is an exact identity with sample statistics, and close to the stated parameters
    ok_exact, ok_close = True, True
    for w in (0.0, 0.25, 0.5, 0.8, 1.0):
        pm, pv = L.sampled_portfolio(r, w)
        v1, v2 = np.var(r[:, 0]), np.var(r[:, 1])
        cov = np.cov(r.T, bias=True)[0, 1]
        ident = w ** 2 * v1 + (1 - w) ** 2 * v2 + 2 * w * (1 - w) * cov
        ok_exact &= abs(ident - pv) < 1e-12
        ok_close &= abs(pv - L.port_var(w, 0.3)) / L.port_var(w, 0.3) < 0.02 and abs(pm - L.port_return(w)) < 0.005
    checks.append(("sampled portfolio variance equals the formula built from sample variances and covariance (exact identity)", ok_exact))
    checks.append(("sampled portfolio mean/variance match the formula within 0.5 points and 2 percent", ok_close))
    pm, pv = L.sampled_portfolio(r, 0.5)
    checks.append((f"sampled w = 0.5 portfolio: return 0.0800 and std 0.1245 (got {pm:.4f}, {np.sqrt(pv):.4f})",
                   abs(pm - 0.0800) < 5e-5 and abs(np.sqrt(pv) - 0.1245) < 5e-5))

    # edge: all in one asset reproduces that asset
    checks.append(("edge: w = 1 gives asset 1 (return 0.10, std 0.20) and w = 0 gives asset 2 (0.06, 0.10)",
                   abs(L.port_std(1.0, 0.3) - 0.2) < 1e-12 and abs(L.port_std(0.0, 0.3) - 0.1) < 1e-12
                   and L.port_return(1.0) == 0.10 and L.port_return(0.0) == 0.06))
    # rho = 1: no benefit, std is the weighted average of the stds
    ws = np.linspace(0, 1, 11)
    checks.append(("rho = 1: risk is the weighted average of the two risks (no benefit)",
                   np.allclose(L.port_std(ws, 1.0), ws * 0.2 + (1 - ws) * 0.1, atol=1e-12)))
    checks.append(("lower rho never raises the risk: std(rho=1) >= std(0.3) >= std(0) >= std(-1) for every w",
                   all(a >= b - 1e-12 >= c - 1e-12 >= d - 1e-12 for a, b, c, d in zip(*[L.port_std(ws, q) for q in (1.0, 0.3, 0.0, -1.0)]))))
    # rho = -1: zero variance at w = s2 / (s1 + s2) = 1/3
    wz = L.zero_variance_weight()
    checks.append(("rho = -1: the weight s2/(s1+s2) = 1/3 gives zero variance", abs(wz - 1 / 3) < 1e-12 and L.port_var(wz, -1.0) < 1e-15))
    checks.append(("rho = -1: the zero-variance mix still earns 0.0733 (= 1/3 * 0.10 + 2/3 * 0.06)", abs(L.port_return(wz) - 0.0733333333) < 1e-9))
    # minimum variance weight: formula vs scan
    ok = all(abs(L.min_var_weight(q) - L.scan_min_var_weight(q)) < 1e-4 for q in (0.0, 0.3, -0.5, -1.0))
    checks.append(("minimum-variance weight w* matches a scan of 100001 weights for rho = 0, 0.3, -0.5, -1", ok))
    checks.append(("w* is 0.2000 at rho = 0, 0.1053 at rho = 0.3, 0.2857 at rho = -0.5",
                   abs(L.min_var_weight(0.0) - 0.2) < 5e-5 and abs(L.min_var_weight(0.3) - 0.1053) < 5e-5
                   and abs(L.min_var_weight(-0.5) - 0.2857) < 5e-5))
    checks.append(("risk at w* is 0.0894 (rho = 0), 0.0979 (rho = 0.3), 0.0655 (rho = -0.5)",
                   abs(L.port_std(L.min_var_weight(0.0), 0.0) - 0.0894) < 5e-5
                   and abs(L.port_std(L.min_var_weight(0.3), 0.3) - 0.0979) < 5e-5
                   and abs(L.port_std(L.min_var_weight(-0.5), -0.5) - 0.0655) < 5e-5))
    checks.append(("at rho = 0 the minimum-risk mix (w = 0.2) earns 0.068 with variance 0.008",
                   abs(L.port_return(0.2) - 0.068) < 1e-12 and abs(L.port_var(0.2, 0.0) - 0.008) < 1e-12))
    checks.append(("edge: equal risks s1 = s2 give w* = 0.5 whatever rho", all(abs(L.min_var_weight(q, 0.15, 0.15) - 0.5) < 1e-12 for q in (-0.9, 0.0, 0.7))))
    checks.append(("edge: rho = 1 gives w* = -1 (outside 0..1), so within 0..1 the safest is all asset 2 (std 0.10)",
                   abs(L.min_var_weight(1.0) + 1.0) < 1e-12 and abs(L.scan_min_var_weight(1.0)) < 1e-12))
    # design problem
    wd = L.weight_for_return(0.07)
    checks.append(("design: a 7% target return needs w = 0.25; at rho = 0.3 its risk is 0.1019",
                   abs(wd - 0.25) < 1e-12 and abs(L.port_std(wd, 0.3) - 0.1019) < 5e-5))
    rmax = L.max_rho_for_risk(0.5, 0.11)
    checks.append(("design: an 8% return (w = 0.5) with risk at most 11% needs rho <= -0.04",
                   abs(rmax + 0.04) < 1e-9 and abs(L.port_std(0.5, rmax) - 0.11) < 1e-9))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
